# 16S sequencing-region check: validating metadata with actual ASV mapping positions

## Goal

These 16S datasets come from NCBI and were processed with **DADA2** to obtain per-study (BioProject) **ASV** sequences. The metadata usually claims which variable region each study targeted (e.g. `V3-V4`), but those records are not always reliable.

The goal of this notebook is to **infer the actually sequenced variable region from the data itself** (the mapping position of each ASV on the E. coli 16S reference), rather than trusting the metadata, and then cross-check against the metadata record to flag studies that agree or disagree.

## Approach in brief

Upstream, each ASV was aligned with `blastn` against the full-length E. coli 16S reference (GenBank **J01695.2**); the result is one `asv_coords.tsv` per study (with `sstart / send / pident / length / evalue`, etc.). Starting from that, this notebook does:

<svg viewBox="0 0 600 360" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif"><defs><marker id="ah" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M0,0 L6,3 L0,6 Z" fill="#4a79b8"/></marker></defs><rect x="20" y="6" width="560" height="46" rx="7" fill="#eef4fb" stroke="#4a79b8" stroke-width="1.5"/><text x="300.0" y="21.0" text-anchor="middle" font-size="13" font-weight="bold" fill="#1c3a5e">DADA2 results per BioProject</text><text x="300.0" y="40.0" text-anchor="middle" font-size="10" fill="#3a5a80">ASVs.fa  ·  asv_coords.tsv (blastn vs E.coli 16S)  ·  DADA2.biom</text><line x1="300.0" y1="52" x2="300.0" y2="74" stroke="#4a79b8" stroke-width="1.5" marker-end="url(#ah)"/><rect x="20" y="78" width="560" height="46" rx="7" fill="#eef4fb" stroke="#4a79b8" stroke-width="1.5"/><text x="300.0" y="93.0" text-anchor="middle" font-size="13" font-weight="bold" fill="#1c3a5e">QC filter + orientation fix</text><text x="300.0" y="112.0" text-anchor="middle" font-size="10" fill="#3a5a80">evalue ≤ 1e-10, length ≥ 70, pident ≥ 60 ; force sstart &lt; send</text><line x1="300.0" y1="124" x2="300.0" y2="146" stroke="#4a79b8" stroke-width="1.5" marker-end="url(#ah)"/><rect x="20" y="150" width="560" height="46" rx="7" fill="#eef4fb" stroke="#4a79b8" stroke-width="1.5"/><text x="300.0" y="165.0" text-anchor="middle" font-size="13" font-weight="bold" fill="#1c3a5e">Each ASV → assign_region → V-region label</text><text x="300.0" y="184.0" text-anchor="middle" font-size="10" fill="#3a5a80">assigned by overlap fraction with each variable region</text><line x1="300.0" y1="196" x2="300.0" y2="218" stroke="#4a79b8" stroke-width="1.5" marker-end="url(#ah)"/><rect x="20" y="222" width="560" height="46" rx="7" fill="#eef4fb" stroke="#4a79b8" stroke-width="1.5"/><text x="300.0" y="237.0" text-anchor="middle" font-size="13" font-weight="bold" fill="#1c3a5e">Aggregate three metrics per study</text><text x="300.0" y="256.0" text-anchor="middle" font-size="10" fill="#3a5a80">seq_region (mode region) · region_prop (proportion) · region_cov_median (vs metadata interval)</text><line x1="300.0" y1="268" x2="300.0" y2="290" stroke="#4a79b8" stroke-width="1.5" marker-end="url(#ah)"/><rect x="20" y="294" width="560" height="46" rx="7" fill="#eef4fb" stroke="#4a79b8" stroke-width="1.5"/><text x="300.0" y="309.0" text-anchor="middle" font-size="13" font-weight="bold" fill="#1c3a5e">Merge with metadata → metadata_pick</text><text x="300.0" y="328.0" text-anchor="middle" font-size="10" fill="#3a5a80">compare seq_region vs sequencing_region_cleaned → match / mismatch</text></svg>

Each study ends up with **two independent lines of evidence**:

1. **`seq_region` + `region_prop`** — measures **internal consistency**: whether the vast majority of ASVs point to the same variable region.
2. **`region_cov_median`** — measures **agreement with the metadata-claimed interval**: whether the ASV positions actually fall inside the region the metadata claims.

These are two different axes: one high and the other low typically means "the data are internally consistent, but disagree with the metadata".

In [111]:
import biom
import pandas as pd
import numpy as np
from Bio import SeqIO
import argparse

## Step 1: variable-region coordinate table + region-assignment function `assign_region`

All variable regions (V1–V9) are given in **E. coli 16S numbering**; the coordinates are taken from **Chakravorty et al. 2007** (*J. Microbiol. Methods* 69(2):330–339), whose reference sequence is J01695.2. `V_REGIONS` below is that coordinate table.

`assign_region` decides which variable regions an ASV (alignment interval $[l,h]$, already min/max) falls into. For each region $r=[s_r,e_r]$ it first computes the overlap between the ASV's alignment interval $A=[a_1,a_2]$ (i.e. $[l,h]$) and a variable region's coordinates $B=[b_1,b_2]$ (i.e. $[s_r,e_r]$):

$$\mathrm{ov}(A,B)=\max\!\big(0,\ \min(a_2,b_2)-\max(a_1,b_1)+1\big)$$

and then the fraction of that region covered by the ASV

$$\mathrm{frac}_r=\frac{\mathrm{ov}\big([l,h],[s_r,e_r]\big)}{e_r-s_r+1}$$

The ASV is considered to "cover" region $r$ when $\mathrm{frac}_r \ge \texttt{min\_frac}$ (default 0.1). The label joins the first and last covered regions (e.g. covering both V3 and V4 → `V3-V4`). If no region is covered, it returns `conserved_only`.

In [119]:
# Chakravorty et al. 2007 V-region coordinates (E. coli numbering, J01695.2)
# ref : A detailed analysis of 16S ribosomal RNA gene segments for the diagnosis of pathogenic bacteria
V_REGIONS = [
    ("V1",   69,   99),
    ("V2",  137,  242),
    ("V3",  433,  497),
    ("V4",  576,  682),
    ("V5",  822,  879),
    ("V6",  986, 1043),
    ("V7", 1117, 1173),
    ("V8", 1243, 1294),
    ("V9", 1435, 1465),
]

def _overlap(a1, a2, b1, b2):
    return max(0, min(a2, b2) - max(a1, b1) + 1)

def assign_region(sstart, send, min_frac=0.10):
    lo, hi = sorted((int(sstart), int(send)))
    hits = []
    for name, rs, re in V_REGIONS:
        frac = _overlap(lo, hi, rs, re) / (re - rs + 1)
        if frac >= min_frac:
            hits.append((name, frac))
    if not hits:
        return "conserved_only", ""      # amplicon sits only in conserved regions
    names = [h[0] for h in hits]
    label = names[0] if len(names) == 1 else f"{names[0]}-{names[-1]}"
    detail = ";".join(f"{n}:{f:.0%}" for n, f in hits)
    return label, detail

## Step 2: load the study list

`project_list` holds all the BioProject accessions to process; the loop below handles them one by one.

In [160]:
project_list = pd.read_csv("../../data/project_list.txt", header=None).iloc[:, 0].values

## Step 3: per study, per ASV — compute the three metrics

For each study in `project_list`:

1. **Read** that study's `ASVs.fa` (ASV id → sequence) and `asv_coords.tsv` (BLAST result).
2. **QC filter** low-quality hits: keep alignments with $\text{evalue}\le 10^{-10}$, $\text{length}\ge 70$, $\text{pident}\ge 60$.
   > Note the `pident` cutoff is only 60: the reference is a single E. coli 16S, while ASVs come from many taxa, so a perfectly good 16S can still be only 70–85% identical to E. coli. **A low pident is normal here**, so no high-identity threshold is used — pident only serves as a floor to drop off-target / non-16S hits.
3. **Orientation fix**: some ASVs align to the minus strand (`sstart > send`); force `sstart` to be the smaller coordinate and `send` the larger.
4. **Keep only ASVs present in the biom table** (align with the actual abundance table).
5. **`region_prop` (internal consistency)**: assign each ASV to one region, take the most frequent one as `seq_region`, and its proportion as `region_prop`:

$$c_r=\#\{a:\ \mathrm{region}(a)=r\},\qquad
\texttt{seq\_region}=\underset{r}{\arg\max}\, c_r,\qquad
\texttt{region\_prop}=\frac{\max_r c_r}{N}$$

where $N$ is the number of ASVs passing QC. The closer `region_prop` is to 1, the more consistently the study's ASVs point to a single region.

6. **`region_cov_median` (agreement with metadata)**: let the metadata-recorded region, converted to coordinates, be $M=[m_{lo},m_{hi}]$. For each ASV (interval $[s_a,e_a]$) compute the fraction of it that falls inside $M$, then take the median over the study:

$$\mathrm{cov}_a=\frac{\mathrm{ov}\big([s_a,e_a],\,[m_{lo},m_{hi}]\big)}{e_a-s_a+1},\qquad
\texttt{region\_cov\_median}=\underset{a}{\mathrm{median}}\;\mathrm{cov}_a$$

The closer `region_cov_median` is to 1, the better the ASV positions match the metadata-claimed interval. **When the metadata is NaN or lists multiple regions (comma-separated), the study is excluded from this metric** (recorded as NaN, `n_asv_used=0`) — because `region_to_interval` returns `None` for both cases.

The figure below illustrates the per-ASV coverage idea (yellow = variable regions, blue band = metadata interval $M$):

<svg viewBox="0 0 720 300" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif"><line x1="55" y1="60" x2="695" y2="60" stroke="#888" stroke-width="2"/><text x="55" y="34" font-size="12" fill="#333" font-weight="bold">E. coli 16S (J01695.2)  1 – 1542 bp</text><line x1="55.41504539559014" y1="56" x2="55.41504539559014" y2="64" stroke="#888"/><text x="55.41504539559014" y="52" text-anchor="middle" font-size="9" fill="#888">1</text><line x1="158.76134889753567" y1="56" x2="158.76134889753567" y2="64" stroke="#888"/><text x="158.76134889753567" y="52" text-anchor="middle" font-size="9" fill="#888">250</text><line x1="262.52269779507134" y1="56" x2="262.52269779507134" y2="64" stroke="#888"/><text x="262.52269779507134" y="52" text-anchor="middle" font-size="9" fill="#888">500</text><line x1="366.284046692607" y1="56" x2="366.284046692607" y2="64" stroke="#888"/><text x="366.284046692607" y="52" text-anchor="middle" font-size="9" fill="#888">750</text><line x1="470.0453955901427" y1="56" x2="470.0453955901427" y2="64" stroke="#888"/><text x="470.0453955901427" y="52" text-anchor="middle" font-size="9" fill="#888">1000</text><line x1="573.8067444876783" y1="56" x2="573.8067444876783" y2="64" stroke="#888"/><text x="573.8067444876783" y="52" text-anchor="middle" font-size="9" fill="#888">1250</text><line x1="695.0" y1="56" x2="695.0" y2="64" stroke="#888"/><text x="695.0" y="52" text-anchor="middle" font-size="9" fill="#888">1542</text><rect x="83.63813229571984" y="51" width="12.451361867704279" height="18" fill="#f2b134" stroke="#c98a00" stroke-width="0.8"/><text x="89.86381322957197" y="82" text-anchor="middle" font-size="9.5" fill="#8a5d00">V1</text><rect x="111.86121919584954" y="51" width="43.57976653696497" height="18" fill="#f2b134" stroke="#c98a00" stroke-width="0.8"/><text x="133.65110246433204" y="82" text-anchor="middle" font-size="9.5" fill="#8a5d00">V2</text><rect x="234.71465629053176" y="51" width="26.56290531776915" height="18" fill="#f2b134" stroke="#c98a00" stroke-width="0.8"/><text x="247.99610894941634" y="82" text-anchor="middle" font-size="9.5" fill="#8a5d00">V3</text><rect x="294.0661478599222" y="51" width="43.99481193255514" height="18" fill="#f2b134" stroke="#c98a00" stroke-width="0.8"/><text x="316.0635538261997" y="82" text-anchor="middle" font-size="9.5" fill="#8a5d00">V4</text><rect x="396.16731517509726" y="51" width="23.657587548638162" height="18" fill="#f2b134" stroke="#c98a00" stroke-width="0.8"/><text x="407.99610894941634" y="82" text-anchor="middle" font-size="9.5" fill="#8a5d00">V5</text><rect x="464.2347600518807" y="51" width="23.657587548638105" height="18" fill="#f2b134" stroke="#c98a00" stroke-width="0.8"/><text x="476.0635538261997" y="82" text-anchor="middle" font-size="9.5" fill="#8a5d00">V6</text><rect x="518.6057068741893" y="51" width="23.24254215304802" height="18" fill="#f2b134" stroke="#c98a00" stroke-width="0.8"/><text x="530.2269779507133" y="82" text-anchor="middle" font-size="9.5" fill="#8a5d00">V7</text><rect x="570.9014267185473" y="51" width="21.167315175097315" height="18" fill="#f2b134" stroke="#c98a00" stroke-width="0.8"/><text x="581.485084306096" y="82" text-anchor="middle" font-size="9.5" fill="#8a5d00">V8</text><rect x="650.5901426718548" y="51" width="12.451361867704236" height="18" fill="#f2b134" stroke="#c98a00" stroke-width="0.8"/><text x="656.8158236057069" y="82" text-anchor="middle" font-size="9.5" fill="#8a5d00">V9</text><text x="695" y="82" text-anchor="end" font-size="9" fill="#aaa">(yellow = variable regions; rest = conserved)</text><rect x="234.71465629053176" y="104" width="103.34630350194556" height="150" fill="#4a79b8" opacity="0.10"/><line x1="234.71465629053176" y1="104" x2="234.71465629053176" y2="254" stroke="#4a79b8" stroke-dasharray="4 3" stroke-width="1"/><line x1="338.0609597924773" y1="104" x2="338.0609597924773" y2="254" stroke="#4a79b8" stroke-dasharray="4 3" stroke-width="1"/><text x="286.3878080415045" y="98" text-anchor="middle" font-size="11" font-weight="bold" fill="#2c4d78">metadata interval M (example V3-V4 = 433–682)</text><rect x="241.7704280155642" y="126" width="87.15953307392996" height="13" rx="3" fill="#3a9d5d"/><text x="235.7704280155642" y="137" text-anchor="end" font-size="9.5" fill="#333">[450,660]</text><text x="336.92996108949416" y="137" font-size="10" fill="#333">cov = 1.00  (fully inside M)</text><rect x="202.75616083009078" y="160" width="68.06744487678344" height="13" rx="3" fill="#e0902b"/><text x="196.75616083009078" y="171" text-anchor="end" font-size="9.5" fill="#333">[356,520]</text><text x="278.8236057068742" y="171" font-size="10" fill="#333">cov ≈ 0.53  (partial overlap)</text><rect x="428.5408560311284" y="194" width="53.95590142671858" height="13" rx="3" fill="#c0504d"/><text x="422.5408560311284" y="205" text-anchor="end" font-size="9.5" fill="#333">[900,1030]</text><text x="490.49675745784697" y="205" font-size="10" fill="#333">cov = 0     (outside M)</text><text x="55" y="240" font-size="10.5" fill="#555">per-ASV coverage = overlap of ASV with M ÷ ASV length ; median over all ASVs = region_cov_median</text></svg>

> Why normalize by the ASV's own length rather than by the metadata interval length: the latter is systematically low because the metadata interval is wider than a single ASV. With the former, an ASV fully inside the claimed interval gets $\mathrm{cov}_a=1$, with no penalty.

In [161]:
V_COORDS = {"V1":(69,99),"V2":(137,242),"V3":(433,497),"V4":(576,682),
            "V5":(822,879),"V6":(986,1043),"V7":(1117,1173),
            "V8":(1243,1294),"V9":(1435,1465)}

def region_to_interval(label):
    if not isinstance(label, str):
        return None
    toks = [t.strip().upper() for t in label.split("-") if t.strip()]
    if not toks or any(t not in V_COORDS for t in toks):   # multi-region with commas also returns None here
        return None
    lo, hi = V_COORDS[toks[0]][0], V_COORDS[toks[-1]][1]
    return (min(lo, hi), max(lo, hi))

# Load metadata first, build a bioproject -> recorded region lookup
metadata = pd.read_csv("../../data/metadata-v2-merge-cleaned.csv", low_memory=False)
meta_region = (metadata.drop_duplicates("bioproject")
                       .set_index("bioproject")["sequencing_region_cleaned"].to_dict())

seq_region, region_prop, region_cov_median, n_asv_used = [], [], [], []

for i in project_list:
    asv_dict   = SeqIO.to_dict(SeqIO.parse(f"/beegfs/dongbiao/oral/{i}/results/ASVs.fa", "fasta"))
    asv_coords = pd.read_csv(f"/beegfs/dongbiao/oral/{i}/results/asv_coords.tsv", sep="\t")

    # QC filter (this was computed but never applied in your version)
    keep = ((asv_coords["evalue"] <= 1e-10) &
            (asv_coords["length"] >= 70) &
            (asv_coords["pident"] >= 60))
    asv_coords = asv_coords[keep].copy()

    # Normalize orientation: some hits are on the minus strand (sstart > send).
    # Force sstart to be the smaller coordinate and send the larger one.
    smin = asv_coords[["sstart", "send"]].min(axis=1)
    smax = asv_coords[["sstart", "send"]].max(axis=1)
    asv_coords["sstart"] = smin
    asv_coords["send"]   = smax

    # Recover sequences, keep only ASVs present in the biom table
    asv_coords["seq"] = [str(asv_dict[a].seq) for a in asv_coords.ASV_seq.values]
    table = biom.load_table(f"/beegfs/dongbiao/oral/{i}/results/DADA2.biom")
    fid = set(table.ids(axis="observation"))
    asv_coords = asv_coords[asv_coords["seq"].isin(fid)].copy()

    # region_prop: most frequent region and its proportion
    reg = asv_coords.apply(lambda r: assign_region(r["sstart"], r["send"], min_frac=0.1)[0], axis=1)
    if len(reg):
        vals, cnts = np.unique(reg.values, return_counts=True)
        j = np.argmax(cnts)
        seq_region.append(vals[j]); region_prop.append(cnts[j] / len(reg))
    else:
        seq_region.append(None); region_prop.append(np.nan)

    # Per-ASV coverage against the metadata interval, then take the median
    label = meta_region.get(i)
    m = region_to_interval(label)          # None for NaN / multi-region
    if m is None or len(asv_coords) == 0:
        region_cov_median.append(np.nan); n_asv_used.append(0)
    else:
        m_lo, m_hi = m
        start = asv_coords["sstart"].values   # already the smaller coordinate
        end   = asv_coords["send"].values     # already the larger coordinate
        overlap = np.maximum(0, np.minimum(end, m_hi) - np.maximum(start, m_lo) + 1)
        cov = overlap / (end - start + 1)     # fraction of each ASV that falls within the metadata interval
        region_cov_median.append(float(np.median(cov)))
        n_asv_used.append(len(cov))

asv_region_df = pd.DataFrame({
    "bioproject":        project_list,
    "seq_region":        seq_region,
    "region_prop":       region_prop,
    "region_cov_median": region_cov_median,   # median of per-ASV coverage for each study
    "n_asv_used":        n_asv_used,           # number of ASVs used (0 = excluded)
})

In [162]:
asv_region_df.shape

(334, 5)

In [163]:
asv_region_df.head()

,bioproject,seq_region,region_prop,region_cov_median,n_asv_used
0,PRJNA362687,V3-V4,0.999305,0.713755,7193
1,PRJNA664107,V3-V4,0.905081,0.705628,2086
2,PRJNA255332,V1,1.000000,0.316327,947
3,PRJEB61123,V3-V4,0.920119,0.710638,13808
4,PRJEB14536,V1-V2,0.643831,0.875000,12393


## Step 4: merge with metadata to build the check table `metadata_pick`

Merge the previous `asv_region_df` with the metadata on `bioproject`, select the columns of interest, and drop duplicates to get `metadata_pick`. **Each column means:**

| Column | Meaning | How it is derived / value range |
|---|---|---|
| `bioproject` | Study (BioProject) accession | from `project_list` |
| `seq_region` | Mode variable region **inferred from the data** (region where most ASVs fall) | $\arg\max_r c_r$, e.g. `V3-V4` |
| `region_prop` | Proportion of ASVs in the mode region — measures **internal consistency** | $\dfrac{\max_r c_r}{N}\in[0,1]$; closer to 1 = more consistent |
| `region_cov_median` | Median over ASVs of "fraction inside the metadata interval" — measures **agreement with metadata** | $\in[0,1]$; higher = better match; `NaN` when metadata is missing / multi-region |
| `n_asv_used` | Number of ASVs used for `region_cov_median` | integer; `0` = study excluded (metadata missing or multi-region) |
| `platform_cleaned` | Sequencing platform (cleaned) | e.g. `Illumina MiSeq` / `Ion Torrent` |
| `sequencing_region_cleaned` | Region **claimed/recorded** in the metadata (cleaned) | e.g. `V3-V4`; may be `NaN` or comma-separated multi-region |

In one line: `seq_region` / `region_prop` only describe **whether the data are internally consistent** (independent of metadata); `region_cov_median` is the **data-vs-metadata** agreement.

In [164]:
metadata = pd.read_csv("../../data/metadata-v2-merge-cleaned.csv", low_memory=False)
metadata = pd.merge(asv_region_df, metadata, on="bioproject", how="inner")
metadata_pick = metadata[['bioproject', 'seq_region', 'region_prop', 'region_cov_median', 'n_asv_used', 'platform_cleaned', 'sequencing_region_cleaned']]
metadata_pick = metadata_pick.drop_duplicates(subset=['bioproject'])

In [170]:
metadata_pick.to_csv("asv_region_res.csv", index=None)

In [165]:
metadata_pick.shape

(334, 7)

In [166]:
metadata_pick.head()

,bioproject,seq_region,region_prop,region_cov_median,n_asv_used,platform_cleaned,sequencing_region_cleaned
0,PRJNA362687,V3-V4,0.999305,0.713755,7193,Illumina MiSeq,V3-V4
111,PRJNA664107,V3-V4,0.905081,0.705628,2086,Illumina HiSeq,V3-V4
157,PRJNA255332,V1,1.000000,0.316327,947,Ion Torrent,V1
181,PRJEB61123,V3-V4,0.920119,0.710638,13808,Illumina MiSeq,V3-V4
253,PRJEB14536,V1-V2,0.643831,0.875000,12393,Illumina MiSeq,V1-V3


## Step 5: decide match / mismatch

- `match`: studies where `seq_region` and `sequencing_region_cleaned` are **exactly equal as strings**.
- `mismatch`: studies where they differ, sorted by `region_prop` for easier triage.

**Do not judge by label equality alone** — combine the two numeric metrics:

- `mismatch` with high `region_prop` and low `region_cov_median` → the data are internally consistent but genuinely disagree with the metadata; the **metadata is probably wrong**, and the mapping result can be preferred.
- `mismatch` but high `region_cov_median` → most likely just a **labeling difference** (e.g. data says `V3`, metadata says `V3-V4`, but the ASVs all fall inside the claimed interval); not a real conflict.
- Low `region_prop` → the study's ASVs are inconsistent themselves (possibly mixed primer sets / many low-quality ASVs); drill down to the run level.

> Note: `match/mismatch` here is a **strict string equality**, so `V3-V4` and `V4` are counted as a mismatch even when they are compatible. It is therefore better to treat `region_cov_median` as a continuous agreement score and bin it, rather than relying on exact label equality.

In [167]:
mismatch = metadata_pick[metadata_pick["seq_region"] != metadata_pick["sequencing_region_cleaned"]]

In [169]:
mismatch.sort_values("region_prop")

,bioproject,seq_region,region_prop,region_cov_median,n_asv_used,platform_cleaned,sequencing_region_cleaned
86996,PRJNA897843,V6-V7,0.186580,NaN,0,Ion Torrent,NaN
94991,PRJNA659632,V6-V7,0.200676,NaN,0,Ion Torrent,"V2,V3,V4,V6-V7,V8,V9"
109196,PRJNA48333,V3-V4,0.222352,0.095238,84083,454 GS FLX Titanium,V3-V5
20787,PRJNA841067,V6-V7,0.224670,NaN,0,Ion Torrent,"V2,V3,V4,V6-V7,V8,V9"
7538,PRJNA232543,V3,0.388503,0.917085,1583,454 GS FLX Titanium,V1-V3
...,...,...,...,...,...,...,...
72337,PRJNA714139,V4,0.998702,0.645740,3851,Illumina MiSeq,V3-V4
72069,PRJNA613363,V4,0.999359,0.843066,7800,Illumina MiSeq,V4-V5
21744,PRJEB31199,V4,0.999740,1.000000,3844,Illumina MiSeq,V3-V4
60819,PRJNA684772,V4,0.999745,0.588448,3914,Illumina MiSeq,V3-V4


In [103]:
match = metadata_pick[metadata_pick["seq_region"] == metadata_pick["sequencing_region_cleaned"]].sort_values("region_prop")